# 03 · Fase B — Intervención preliminar: número de object queries
**Antes de ejecutar:** las hipótesis H1–H3 deben estar escritas y commiteadas en `DESIGN_LOG.md` (la celda siguiente imprime el commit como evidencia).
Una hipótesis, un cambio (N), una medición. Mismo protocolo, inicialización de queries (D6), resolución y semilla en todas las corridas.

In [ ]:
# Configuración común (Colab/Kaggle o local). Ajusta REPO_URL una sola vez.
import os, sys
REPO_URL = 'https://github.com/<usuario>/detr-query-saturation.git'
if not os.path.exists('../src/common.py'):            # no estamos dentro de notebooks/ -> clonar
    if not os.path.exists('detr-query-saturation/src'):
        !git clone -q {REPO_URL}
    os.chdir('detr-query-saturation/notebooks')
    !pip -q install -r ../requirements.txt
sys.path.insert(0, '../src')
from common import *
import torch, json, pandas as pd, matplotlib.pyplot as plt
os.makedirs('../results', exist_ok=True); os.makedirs('../data', exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'; SEED = 42; set_seed(SEED)
print(DEVICE); print(env_report('../results/entorno.txt'))   # versiones exactas -> README

In [ ]:
# Evidencia: último commit del design log (debe ser ANTERIOR a las corridas de abajo)
!git -C .. log -1 --format='%h  %ci  %s' -- DESIGN_LOG.md

In [ ]:
# Rutas de la copia de Kaggle de VisDrone (ajusta tras revisar el árbol en 01)
ROOT = '/kaggle/input/<dataset-visdrone>'
TRAIN_IMG, TRAIN_LBL = f'{ROOT}/<train/images>', f'{ROOT}/<train/labels>'
VAL_IMG,   VAL_LBL   = f'{ROOT}/<val/images>',   f'{ROOT}/<val/labels>'

from transformers import DetrImageProcessor, DetrForObjectDetection
CKPT = 'facebook/detr-resnet-50'
processor = DetrImageProcessor.from_pretrained(CKPT, size={'shortest_edge': 512, 'longest_edge': 853})   # D4
TRAIN_JSON, VAL_JSON = '../data/train_sub.json', '../data/val_sub.json'
if not os.path.exists(TRAIN_JSON):                    # por si se abre en otra sesión sin los JSON
    yolo_to_coco(TRAIN_IMG, TRAIN_LBL, '../data/train.json'); make_subset('../data/train.json', 600, TRAIN_JSON, SEED)
    yolo_to_coco(VAL_IMG, VAL_LBL, '../data/val.json');       make_subset('../data/val.json', None, VAL_JSON, SEED)
EPOCHS, BS = 8, 2                                                   # D13
groups, (q1, q2) = density_groups(VAL_JSON); print({k: len(v) for k, v in groups.items()}, 'cortes:', q1, q2)
cnt = objects_per_image(VAL_JSON)
print(pd.DataFrame({g: {N: round(oracle_recall([cnt[i] for i in ids], N), 3) for N in (100, 300, 500)}
                    for g, ids in groups.items()}).rename_axis('techo N'))   # techo analítico por grupo

## B0 · Punto de partida: DETR de COCO sin fine-tuning (zero-shot, N = 100)
Solo las 6 clases con equivalente en COCO (`COCO_TO_VISDRONE`). Es también la primera medición de la Fase C.

In [ ]:
MAPPED = sorted(set(COCO_TO_VISDRONE.values()))
zs_model = DetrForObjectDetection.from_pretrained(CKPT)
zs, zs_dets = evaluate_all(zs_model, processor, VAL_IMG, VAL_JSON, groups=groups, device=DEVICE,
                           label_offset=0, label_map=COCO_TO_VISDRONE, cat_ids=MAPPED, return_dets=True)
del zs_model; torch.cuda.empty_cache()
print(pd.DataFrame(zs).T[['AP', 'AP50', 'AP_S', 'AR@100', 'AR@500', 'n_img']].round(3))
json.dump(zs, open('../results/zero_shot.json', 'w'), indent=1)

## B1 · Ablación (se reanuda si la sesión se cae: salta las N ya hechas)

In [ ]:
import time
VARIANTS = (100, 300)                # añade 500 si hay tiempo (mismo protocolo)
RES_PATH = '../results/ablation_queries.json'
results = {int(k): v for k, v in json.load(open(RES_PATH)).items()} if os.path.exists(RES_PATH) else {}
for N in VARIANTS:
    if N in results: print('ya hecho', N); continue
    set_seed(SEED)
    model = build_model(num_queries=N, num_labels=10, freeze_backbone=True)       # D5, D6
    print(f'N={N}: parámetros {count_params(model)/1e6:.2f} M | entrenables {count_params(model, True)/1e6:.2f} M')
    tl = make_loader(TRAIN_IMG, TRAIN_JSON, processor, bs=BS, shuffle=True, seed=SEED)
    t0 = time.time()
    hist = train(model, tl, EPOCHS, device=DEVICE, ckpt_path=f'/tmp/detr_N{N}.pt')   # checkpoint fuera del repo
    res, dets = evaluate_all(model, processor, VAL_IMG, VAL_JSON, groups=groups, device=DEVICE, return_dets=True)
    res['mapped6'] = coco_eval(VAL_JSON, dets, cat_ids=MAPPED)                      # comparable con el zero-shot
    json.dump(dets, open(f'../results/dets_N{N}.json', 'w'))
    results[N] = {'loss': hist, 'eval': res, 'minutos': round((time.time() - t0) / 60, 1)}
    json.dump(results, open(RES_PATH, 'w'), indent=1)
    print(N, {k: round(v, 3) for k, v in res['overall'].items()}); del model; torch.cuda.empty_cache()

In [ ]:
rows = [{'N': N, 'grupo': g, **m} for N, r in results.items() for g, m in r['eval'].items()]
T = pd.DataFrame(rows); print(T[['N', 'grupo', 'AP', 'AP50', 'AP_S', 'AR@100', 'AR@500', 'n_img']].round(3).to_string())
T.to_csv('../results/ablation_queries.csv', index=False)
order = [g for g in ['low', 'medium', 'high', 'gt100', 'overall'] if g in set(T['grupo'])]
P = T.pivot(index='grupo', columns='N', values='AR@500').loc[order]
P.plot(kind='bar'); plt.ylabel('AR@500'); plt.title('AR@500 por densidad de escena'); plt.xticks(rotation=0)
plt.savefig('../results/ar500_por_densidad.png', dpi=150, bbox_inches='tight')

## B2 · Recall por imagen vs techo analítico min(1, N/n) (H1)
IoU ≥ 0.5, emparejamiento 1 a 1 por clase. Si los puntos de N=100 se pegan a la curva para n > 100, el techo está activo; si quedan muy por debajo, otro factor actúa antes (resolución, objetos pequeños, shift).

In [ ]:
import numpy as np
fig, ax = plt.subplots(figsize=(7, 4.5)); xs = np.arange(1, max(cnt.values()) + 1)
for N in sorted(results):
    R = pd.DataFrame(per_image_recall(VAL_JSON, json.load(open(f'../results/dets_N{N}.json')))).dropna()
    R['bin'] = pd.cut(R['n_gt'], bins=[0, 25, 50, 100, 150, 200, 300, 1000])
    B = R.groupby('bin', observed=True).agg(n=('n_gt', 'mean'), recall=('recall', 'mean'), imgs=('recall', 'size'))
    print(f'N={N}\n', B.round(3)); B.to_csv(f'../results/recall_por_densidad_N{N}.csv')
    l, = ax.plot(B['n'], B['recall'], 'o-', label=f'medido N={N}')
    ax.plot(xs, np.minimum(1, N / xs), '--', color=l.get_color(), alpha=.6, label=f'techo N={N}')
ax.set_xlabel('objetos por imagen (n)'); ax.set_ylabel('recall @ IoU 0.5'); ax.legend(fontsize=8)
plt.savefig('../results/recall_vs_techo.png', dpi=150, bbox_inches='tight')

In [ ]:
for N, r in sorted(results.items()):
    plt.plot([h['epoch'] for h in r['loss']], [h['loss'] for h in r['loss']], 'o-', label=f'N={N} ({r["minutos"]} min)')
plt.xlabel('época'); plt.ylabel('pérdida de entrenamiento'); plt.legend(); plt.savefig('../results/curvas_perdida.png', dpi=150, bbox_inches='tight')

## Contraste con las hipótesis (completar en el design log)
- **H1** (recall cae con N=100 en escenas densas): mirar `gt100`/`high` vs `low` y la figura recall vs techo. ¿Se cumplió? ¿por qué?
- **H2** (más queries ayuda sobre todo en escenas densas): diferencia N=300 − N=100 por grupo. ¿Es mayor en `high` que en `low`? ¿La pérdida converge peor con más queries?
- **H3** (costo bajo): tabla `costo_vs_N.csv` del notebook 02 (A8).

⚠️ Entrenamiento corto (8 épocas) en un subconjunto de 600 imágenes: interpreten **diferencias relativas**, no el AP absoluto. Repórtenlo como limitación.